# 01. Data reality check

**Uplift Modeling and Targeting Policy** &nbsp;|&nbsp; Notebook 1 of 3

---

### The question this project answers

An ad campaign lifted site visits. That does not mean it should have
been sent to everyone. Given a budget that covers only part of the
audience, which users should receive the ad, and does ranking by
predicted uplift beat ranking by predicted response probability or
sending to everyone?

### What this notebook does

Establishes what is in the data and what it can support, before any
modelling decision is made. It computes and prints. It does not
model and it does not sample.

Five things come out of it:

1. The file is what the documentation claims it is
2. The experiment is valid, and where it departs from a textbook
   randomised trial
3. Which outcome can be modelled and which cannot
4. How much apparent effect this dataset produces from nothing, which
   is the bar any model has to clear
5. Whether uplift is measurable across the whole feature space

### Data

Criteo Uplift Prediction dataset, version 2.1. A real randomised
advertising experiment released by Criteo.

Not included in this repository. Download from
`http://go.criteo.net/criteo-research-uplift-v2.1.csv.gz` and place
it in `Data/`. The file is 311MB.

Two versions circulate under the same name. Version 1 has 25,309,483
rows. This project uses version 2.1, with 13,979,592. The hash
printed in section 1 identifies exactly which file produced these
numbers.

Citation: Diemert, Betlei, Renaudin and Amini, *A Large Scale
Benchmark for Uplift Modeling*, AdKDD 2018 workshop at KDD.

### Runtime

About three minutes, plus one minute to load the file. Peak memory
around 2GB.

## Setup

In [1]:
import hashlib
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = Path("../Data/criteo-uplift-v2.1.csv.gz")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No file at {DATA_PATH.resolve()}. Check DATA_PATH."
    )

FEATURES = [f"f{i}" for i in range(12)]
OUTCOMES = ["visit", "conversion"]
TREATMENT_COL = "treatment"
BINARY_COLS = ["treatment", "visit", "conversion", "exposure"]

BIN_COUNTS = [10, 20, 50, 100]
NOISE_SEEDS = [11, 12, 13, 14, 15]
N_PERMUTATIONS = 50
PERMUTATION_SEED = 20260924

PROPENSITY_SAMPLE_ROWS = 5_000_000
PROPENSITY_TEST_SIZE = 0.3
PROPENSITY_SEED = 20260924
N_NULL_DRAWS = 3

Z_95 = 1.959963985

DTYPES = {name: "float32" for name in FEATURES}
DTYPES.update({name: "int8" for name in BINARY_COLS})

LGB_PROPENSITY_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 200,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "n_jobs": -1,
    "random_state": PROPENSITY_SEED,
    "verbose": -1,
}

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print("lightgbm", lgb.__version__)
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)

lightgbm 4.6.0
numpy    2.4.1
pandas   3.0.0


## 1. File identity

The hash and row count go in the README so a reader knows exactly
which file produced every number in this project.

In [3]:
df = pd.read_csv(DATA_PATH, compression="gzip", dtype=DTYPES)
df.shape

(13979592, 16)

In [4]:
def compute_file_hash(path, chunk_size=1 << 20):
    """Return the sha256 hex digest of a file.

    Args:
        path (Path): Path to the file.
        chunk_size (int): Bytes read per iteration.

    Returns:
        str: Hex digest.
    """
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


file_facts = {
    "file_name": DATA_PATH.name,
    "sha256": compute_file_hash(DATA_PATH),
    "size_bytes": DATA_PATH.stat().st_size,
    "n_rows": len(df),
    "columns": list(df.columns),
    "n_missing_total": int(df.isna().sum().sum()),
}

for key, value in file_facts.items():
    print(f"{key}: {value}")

file_name: criteo-uplift-v2.1.csv.gz
sha256: 2716e1bf0fd157a93b5bf86924d9088419dfbac2022c6cd90030220634f616dc
size_bytes: 311422618
n_rows: 13979592
columns: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']
n_missing_total: 0


## 2. What the columns are, and the order they happen in

Every row is one user. The columns record events in a fixed order,
and that order governs what may be used for what.

| When | Column | Meaning |
|---|---|---|
| Before | `f0` to `f11` | Twelve anonymised numbers describing the user |
| Then | `treatment` | Randomly assigned. Eligible to see the ad, or blocked |
| Then | `exposure` | Whether an ad was actually served |
| Then | `visit` | Whether the user visited the advertiser's site |
| Last | `conversion` | Whether the user bought something |

Only `treatment` was assigned by chance. That is what makes causal
comparison possible, and it applies to nothing else in the file.

`exposure`, `visit` and `conversion` were all determined by user
behaviour after assignment. None of them may be used as a model
input or to define comparison groups. Section 2b shows what happens
if that rule is broken.

### 2a. Binary columns

In [5]:
def check_binary_columns(frame, columns):
    """Check that the given columns contain only zeros and ones.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        columns (list): Column names expected to be binary.

    Returns:
        pandas.DataFrame: Observed unique values per column.
    """
    rows = []
    for name in columns:
        values = sorted(frame[name].unique().tolist())
        rows.append({
            "column": name,
            "unique_values": values,
            "is_binary": set(values).issubset({0, 1}),
        })
    return pd.DataFrame(rows)


check_binary_columns(df, BINARY_COLS)

,column,unique_values,is_binary
0,treatment,"[0, 1]",True
1,visit,"[0, 1]",True
2,conversion,"[0, 1]",True
3,exposure,"[0, 1]",True


### 2b. Exposure, and why it is excluded

The documentation says control users were blocked from seeing ads.
If that held, no control row can have `exposure = 1`. The cell for
treatment 0 and exposure 1 should be empty.

In [6]:
pd.crosstab(df[TREATMENT_COL], df["exposure"])

exposure,0,1
treatment,,
0,2096937,0
1,11454443,428212


Now the same column inside the treated arm only, against `visit`.
This is the evidence that `exposure` cannot be used for comparison.

In [7]:
treated = df[df[TREATMENT_COL] == 1]
control = df[df[TREATMENT_COL] == 0]

exposure_by_visit = pd.crosstab(
    treated["exposure"], treated["visit"])

rows = []
for flag, label in [(0, "never shown an ad"), (1, "shown an ad")]:
    subset = treated[treated["exposure"] == flag]
    rows.append({
        "group": label,
        "users": len(subset),
        "visits": int(subset["visit"].sum()),
        "visit_rate": subset["visit"].mean(),
    })
rows.append({
    "group": "control arm, for comparison",
    "users": len(control),
    "visits": int(control["visit"].sum()),
    "visit_rate": control["visit"].mean(),
})

display(exposure_by_visit)
pd.DataFrame(rows)

visit,0,1
exposure,,
0,11055129,399314
1,250702,177510


,group,users,visits,visit_rate
0,never shown an ad,11454443,399314,0.034861
1,shown an ad,428212,177510,0.414538
2,"control arm, for comparison",2096937,80105,0.038201


Two things fall out of that table.

**The trap, made concrete.** Comparing users who were shown an ad
against the control arm gives a difference of roughly 37.6 percentage
points. The correct intent-to-treat figure, computed in section 4, is
1.03 percentage points. The naive comparison overstates the effect by
a factor of about 36.

**The mechanism, made visible.** Users who were eligible but never
shown an ad received no advertising at all, exactly like the control
arm. They visit at a *lower* rate than the control arm. The ad did
not make them visit less. Removing the heavy browsers from the
treated arm left a less active remainder.

Exposure is caused by user behaviour, so exposed users are a
self-selected population of heavy browsers who visit more often
regardless of advertising. Every comparison in this project is
therefore **intent to treat**: users are compared by the arm they
were assigned to, whatever happened afterwards.

One consequence carried forward. Only 3.6% of eligible users were
ever shown an ad, so if the advertiser is billed per impression, a
targeted user costs about 0.036 impressions rather than one.
Notebook 3 depends on this.

### 2c. Observed allocation

A sample ratio mismatch test compares observed allocation against a
pre-specified designed allocation. No design document is published
with this dataset. The 0.85 figure is a rounded summary reported
after the fact, not the target the randomiser was given. At 14M rows
a chi-square test against exactly 0.85 rejects on a rounding
difference and tells you nothing.

Observed share is reported. No test is run.

In [8]:
allocation = df[TREATMENT_COL].value_counts().rename(
    "rows").to_frame()
allocation["share"] = df[TREATMENT_COL].value_counts(normalize=True)
allocation.index = ["treated" if i == 1 else "control"
                    for i in allocation.index]
allocation

,rows,share
treated,11882655,0.850000
control,2096937,0.150000


### 2d. Covariate balance

The causal claim rests on the arms being alike before the campaign.
That is checkable directly on the twelve features.

Raw mean differences are uninterpretable because the features are on
unknown scales. The standard measure is the **standardised mean
difference**, which expresses the gap in units of the feature's own
spread:

```
smd = (mean_treated - mean_control) / pooled_standard_deviation
```

The usual convention treats absolute SMD below 0.1 as negligible.

No t-tests. With 2.1M control rows a t-test returns significance on
differences far too small to matter, so magnitude has to be judged
directly. Section 7 supplies the correct comparison point.

In [9]:
def standardised_mean_difference(frame, features):
    """Compute the standardised mean difference for each feature.

    Uses the pooled standard deviation across arms. Magnitude is the
    quantity of interest; significance testing is not meaningful at
    this sample size.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        features (list): Feature column names.

    Returns:
        pandas.DataFrame: Means by arm and the SMD per feature.
    """
    arm_t = frame[frame[TREATMENT_COL] == 1]
    arm_c = frame[frame[TREATMENT_COL] == 0]

    rows = []
    for name in features:
        mean_t, mean_c = arm_t[name].mean(), arm_c[name].mean()
        var_t, var_c = arm_t[name].var(ddof=1), arm_c[name].var(ddof=1)
        pooled_sd = np.sqrt((var_t + var_c) / 2)
        smd = np.nan if pooled_sd == 0 else (mean_t - mean_c) / pooled_sd
        rows.append({
            "feature": name,
            "mean_treated": mean_t,
            "mean_control": mean_c,
            "smd": smd,
            "abs_smd": abs(smd),
        })
    return pd.DataFrame(rows).sort_values(
        "abs_smd", ascending=False).reset_index(drop=True)


balance = standardised_mean_difference(df, FEATURES)
balance

,feature,mean_treated,mean_control,smd,abs_smd
0,f3,4.169412,4.232821,-0.048836,0.048836
1,f6,-4.182793,-3.999880,-0.040448,0.040448
2,f5,4.026602,4.039338,-0.030557,0.030557
3,f9,16.052589,15.886255,0.024001,0.024001
4,f1,10.070336,10.067936,0.023986,0.023986
5,f8,3.933392,3.934652,-0.022427,0.022427
6,f7,5.105556,5.080285,0.021269,0.021269
7,f10,5.333661,5.331899,0.010565,0.010565
8,f4,10.339246,10.336524,0.007972,0.007972
9,f0,19.614754,19.651705,-0.006866,0.006866


## 3. Arm counts and rates

Raw counts alongside rates. The counts are what determine whether an
estimate is stable, and they are the constraint the rest of the
project works within.

In [10]:
def arm_summary(frame, outcomes):
    """Summarise counts and rates for each arm.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcomes (list): Binary outcome column names.

    Returns:
        pandas.DataFrame: Rows for counts and rates, one column per
        arm.
    """
    arm_t = frame[frame[TREATMENT_COL] == 1]
    arm_c = frame[frame[TREATMENT_COL] == 0]

    summary = {
        "rows": {"treated": len(arm_t), "control": len(arm_c)},
        "share_of_total": {
            "treated": len(arm_t) / len(frame),
            "control": len(arm_c) / len(frame),
        },
    }
    for outcome in outcomes:
        summary[f"{outcome}_positives"] = {
            "treated": int(arm_t[outcome].sum()),
            "control": int(arm_c[outcome].sum()),
        }
        summary[f"{outcome}_rate"] = {
            "treated": float(arm_t[outcome].mean()),
            "control": float(arm_c[outcome].mean()),
        }
    return pd.DataFrame(summary).T


arm_summary(df, OUTCOMES)

,treated,control
rows,"11,882,655.000000","2,096,937.000000"
share_of_total,0.850000,0.150000
visit_positives,"576,824.000000","80,105.000000"
visit_rate,0.048543,0.038201
conversion_positives,"36,711.000000","4,063.000000"
conversion_rate,0.003089,0.001938


### The funnel

Conversion is nested inside visit. Nobody buys without visiting
first, so the campaign's effect on conversion is partly its effect on
visits flowing downstream.

In [11]:
display(pd.crosstab(df["visit"], df["conversion"]))

funnel = pd.DataFrame({
    "visit_rate": {
        "treated": treated["visit"].mean(),
        "control": control["visit"].mean(),
    },
    "conversion_given_visit": {
        "treated": treated.loc[treated["visit"] == 1,
                               "conversion"].mean(),
        "control": control.loc[control["visit"] == 1,
                               "conversion"].mean(),
    },
    "conversion_rate": {
        "treated": treated["conversion"].mean(),
        "control": control["conversion"].mean(),
    },
}).T
funnel["ratio"] = funnel["treated"] / funnel["control"]
funnel

conversion,0,1
visit,,
0,13322663,0
1,616155,40774


,treated,control,ratio
visit_rate,0.048543,0.038201,1.270737
conversion_given_visit,0.063643,0.050721,1.254775
conversion_rate,0.003089,0.001938,1.594488


The three ratios multiply: the visit ratio times the
conversion-given-visit ratio equals the conversion ratio.

One caution. The middle row conditions on `visit`, which is a
post-treatment outcome. Treated visitors and control visitors are not
comparable groups, because the ad changed who became a visitor. So
that ratio is descriptive arithmetic and not a causal effect. It is
reported as a description of the funnel, with the limitation stated
alongside it.

## 4. Average treatment effect

The difference in outcome rates between arms, with a confidence
interval.

```
estimate = rate_treated - rate_control
std_err  = sqrt(p_t(1 - p_t)/n_t + p_c(1 - p_c)/n_c)
ci_95    = estimate +/- 1.96 * std_err
```

Both absolute and relative lift are reported. Absolute lift in
percentage points is what profit is computed from in notebook 3.
Relative lift is what a marketer quotes. Showing both means no reader
has to guess which one a number is.

In [12]:
def difference_in_proportions(frame, outcome):
    """Estimate the average treatment effect for a binary outcome.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcome (str): Binary outcome column name.

    Returns:
        dict: Point estimate, standard error, interval bounds, and
        relative lift over the control rate.
    """
    arm_t = frame.loc[frame[TREATMENT_COL] == 1, outcome]
    arm_c = frame.loc[frame[TREATMENT_COL] == 0, outcome]

    n_t, n_c = len(arm_t), len(arm_c)
    p_t, p_c = arm_t.mean(), arm_c.mean()

    diff = p_t - p_c
    std_err = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
    margin = Z_95 * std_err

    return {
        "outcome": outcome,
        "rate_treated": p_t,
        "rate_control": p_c,
        "ate_absolute": diff,
        "std_error": std_err,
        "ci_lower": diff - margin,
        "ci_upper": diff + margin,
        "ate_relative": diff / p_c if p_c > 0 else np.nan,
        "z_statistic": diff / std_err if std_err > 0 else np.nan,
    }


effects = pd.DataFrame(
    [difference_in_proportions(df, name) for name in OUTCOMES])
effects

,outcome,rate_treated,rate_control,ate_absolute,std_error,ci_lower,ci_upper,ate_relative,z_statistic
0,visit,0.048543,0.038201,0.010342,0.000146,0.010056,0.010629,0.270737,70.685184
1,conversion,0.003089,0.001938,0.001152,0.000034,0.001085,0.001219,0.594488,33.512267


## 5. Which outcome can be modelled

Uplift is a difference between two rates, and the control arm is the
smaller side. Once a model partitions the population into score bins,
the evidence available inside one bin is what determines whether the
estimate means anything.

In [13]:
def control_positives_per_bin(frame, outcomes, bin_counts):
    """Control-arm positives spread over equal-sized bins.

    Equal-sized bins are assumed, so this is an order-of-magnitude
    figure rather than a precise count.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcomes (list): Binary outcome column names.
        bin_counts (list): Bin counts to evaluate.

    Returns:
        pandas.DataFrame: One row per outcome and bin count.
    """
    arm_c = frame[frame[TREATMENT_COL] == 0]

    rows = []
    for outcome in outcomes:
        positives = int(arm_c[outcome].sum())
        for n_bins in bin_counts:
            rows.append({
                "outcome": outcome,
                "n_bins": n_bins,
                "control_positives": positives,
                "positives_per_bin": positives / n_bins,
            })
    return pd.DataFrame(rows)


control_positives_per_bin(df, OUTCOMES, BIN_COUNTS)

,outcome,n_bins,control_positives,positives_per_bin
0,visit,10,80105,"8,010.500000"
1,visit,20,80105,"4,005.250000"
2,visit,50,80105,"1,602.100000"
3,visit,100,80105,801.050000
4,conversion,10,4063,406.300000
5,conversion,20,4063,203.150000
6,conversion,50,4063,81.260000
7,conversion,100,4063,40.630000


## 6. The noise floor

### Why this exists

A model will sort users by predicted uplift, and a chart will show
uplift rising across the sorted groups. That chart looks convincing
whether or not the model found anything.

Uplift in any group is a difference between two estimated rates, and
estimated rates wobble. Sort by any score, even a meaningless one,
and the groups where the wobble happened to go up land at one end.
Sorting turns noise into an apparent trend.

### The measurement

Assign every user a bin number completely at random. Since assignment
ignores everything about the user, the true uplift is identical in
every bin. Measure uplift in each bin anyway.

The spread across bins is a direct measurement of how much fake
heterogeneity this dataset manufactures at this sample size. It is
the bar any model has to clear.

A second quantity comes free. Comparing the observed spread against
what independent rows would produce tests whether the standard error
formula used in section 4 is trustworthy. A ratio near 1 means it is.

In [14]:
def random_bin_uplift(frame, outcome, n_bins, seed):
    """Estimate uplift within randomly assigned bins.

    Bins are assigned independently of features and of treatment, so
    the true uplift is the same in every bin. Any observed spread is
    sampling noise.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcome (str): Binary outcome column name.
        n_bins (int): Number of random bins.
        seed (int): Seed for bin assignment.

    Returns:
        numpy.ndarray: Uplift estimate per bin.
    """
    rng = np.random.default_rng(seed)

    treatment = frame[TREATMENT_COL].to_numpy()
    response = frame[outcome].to_numpy()
    bins = rng.integers(0, n_bins, size=len(frame))

    cell = bins * 2 + treatment
    length = n_bins * 2

    positives = np.bincount(cell, weights=response,
                            minlength=length).reshape(n_bins, 2)
    counts = np.bincount(cell, minlength=length).reshape(n_bins, 2)

    return positives[:, 1] / counts[:, 1] - positives[:, 0] / counts[:, 0]


def analytic_uplift_std_error(frame, outcome, n_bins):
    """Expected standard error of a within-bin uplift estimate.

    Assumes independent rows and equal-sized bins.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcome (str): Binary outcome column name.
        n_bins (int): Number of bins.

    Returns:
        float: Expected standard error under independence.
    """
    arm_t = frame.loc[frame[TREATMENT_COL] == 1, outcome]
    arm_c = frame.loc[frame[TREATMENT_COL] == 0, outcome]

    p_t, p_c = arm_t.mean(), arm_c.mean()
    n_t, n_c = len(arm_t) / n_bins, len(arm_c) / n_bins

    return np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)

In [15]:
def noise_floor(frame, outcomes, bin_counts, seeds):
    """Summarise the spread of random-bin uplift estimates.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        outcomes (list): Binary outcome column names.
        bin_counts (list): Bin counts to evaluate.
        seeds (list): Seeds for repeated bin assignment.

    Returns:
        pandas.DataFrame: One row per outcome and bin count,
        averaged over seeds.
    """
    ates = {name: difference_in_proportions(frame, name)["ate_absolute"]
            for name in outcomes}

    rows = []
    for outcome in outcomes:
        for n_bins in bin_counts:
            analytic = analytic_uplift_std_error(frame, outcome, n_bins)
            spreads, ranges, means = [], [], []
            for seed in seeds:
                values = random_bin_uplift(frame, outcome, n_bins, seed)
                spreads.append(values.std(ddof=1))
                ranges.append(values.max() - values.min())
                means.append(values.mean())
            rows.append({
                "outcome": outcome,
                "n_bins": n_bins,
                "uplift_mean": np.mean(means),
                "ate_absolute": ates[outcome],
                "noise_sd": np.mean(spreads),
                "analytic_sd": analytic,
                "sd_ratio": np.mean(spreads) / analytic,
                "noise_sd_over_ate": np.mean(spreads) / ates[outcome],
                "noise_range_over_ate": np.mean(ranges) / ates[outcome],
            })
    return pd.DataFrame(rows)


noise = noise_floor(df, OUTCOMES, BIN_COUNTS, NOISE_SEEDS)
noise

,outcome,n_bins,uplift_mean,ate_absolute,noise_sd,analytic_sd,sd_ratio,noise_sd_over_ate,noise_range_over_ate
0,visit,10,0.010342,0.010342,0.000502,0.000463,1.084092,0.048500,0.160444
1,visit,20,0.010342,0.010342,0.000648,0.000654,0.990895,0.062692,0.211469
2,visit,50,0.010342,0.010342,0.001087,0.001035,1.050402,0.105078,0.467906
3,visit,100,0.010342,0.010342,0.001470,0.001463,1.005004,0.142180,0.706871
4,conversion,10,0.001152,0.001152,0.000105,0.000109,0.970083,0.091539,0.271855
5,conversion,20,0.001152,0.001152,0.000143,0.000154,0.931308,0.124281,0.480624
6,conversion,50,0.001152,0.001152,0.000252,0.000243,1.036517,0.218705,1.001298
7,conversion,100,0.001152,0.001152,0.000342,0.000344,0.996223,0.297271,1.439504


How to read this.

| Column | Meaning |
|---|---|
| `uplift_mean` | Should equal the overall ATE. If not, the binning code is wrong |
| `noise_sd` | The noise floor. Real heterogeneity must be large relative to this |
| `sd_ratio` | Observed spread over what independent rows predict. Near 1 means the section 4 intervals are trustworthy |
| `noise_range_over_ate` | Above 1 means noise alone can produce a best-to-worst bin gap larger than the entire average effect |

`noise_range_over_ate` is the number that governs how results may be
presented. Where it approaches 1, a decile chart can look informative
while measuring nothing.

## 7. Is the imbalance real? A permutation test

Section 2d found all twelve SMDs below 0.05, which passes the usual
convention. That convention is a rule of thumb, not a property of
this data, and it does not say whether 0.0488 is a normal amount of
imbalance at this sample size.

That question has an exact answer requiring no formula.

Discard the real treatment assignment and redeal it at random,
keeping arm sizes fixed. The result is a world where assignment is
guaranteed to have nothing to do with any feature. Recompute all
twelve SMDs and record the largest. Repeat fifty times.

Those fifty values are what random assignment actually produces on
this data. The real value either sits inside them or it does not.

In [16]:
def permutation_smd_null(frame, features, n_permutations, seed):
    """Build the permutation distribution of maximum absolute SMD.

    Column totals are computed once and the treated arm derived by
    subtraction, so each permutation costs one gather per feature
    rather than a full groupby.

    Args:
        frame (pandas.DataFrame): Loaded dataset.
        features (list): Feature column names.
        n_permutations (int): Number of permutations to draw.
        seed (int): Seed for the permutation sequence.

    Returns:
        pandas.DataFrame: One row per permutation and feature.
    """
    rng = np.random.default_rng(seed)

    n_rows = len(frame)
    n_control = int((frame[TREATMENT_COL] == 0).sum())
    n_treated = n_rows - n_control

    columns, totals = {}, {}
    for name in features:
        values = frame[name].to_numpy()
        columns[name] = values
        wide = values.astype(np.float64)
        totals[name] = (wide.sum(), np.dot(wide, wide))
        del wide

    rows = []
    for draw in range(n_permutations):
        keys = rng.random(n_rows)
        control_index = np.argpartition(keys, n_control)[:n_control]
        del keys

        for name in features:
            subset = columns[name][control_index].astype(np.float64)
            sum_c = subset.sum()
            sumsq_c = np.dot(subset, subset)
            del subset

            sum_all, sumsq_all = totals[name]
            sum_t, sumsq_t = sum_all - sum_c, sumsq_all - sumsq_c

            mean_c, mean_t = sum_c / n_control, sum_t / n_treated
            var_c = (sumsq_c - n_control * mean_c ** 2) / (n_control - 1)
            var_t = (sumsq_t - n_treated * mean_t ** 2) / (n_treated - 1)

            pooled_sd = np.sqrt((var_t + var_c) / 2)
            smd = np.nan if pooled_sd == 0 else (
                (mean_t - mean_c) / pooled_sd)

            rows.append({"permutation": draw, "feature": name,
                         "abs_smd": abs(smd)})

    return pd.DataFrame(rows)


permuted = permutation_smd_null(
    df, FEATURES, N_PERMUTATIONS, PERMUTATION_SEED)
permuted.shape

(600, 3)

In [17]:
null_max = permuted.groupby("permutation")["abs_smd"].max()
observed_max = balance["abs_smd"].max()

analytic_se = np.sqrt(
    1 / (df[TREATMENT_COL] == 1).sum()
    + 1 / (df[TREATMENT_COL] == 0).sum())

pd.Series({
    "observed_max_abs_smd": observed_max,
    "analytic_se_of_smd": analytic_se,
    "observed_in_analytic_se": observed_max / analytic_se,
    "null_max_mean": null_max.mean(),
    "null_max_largest": null_max.max(),
    "permutation_p_value": float((null_max >= observed_max).mean()),
    "n_permutations": len(null_max),
})

observed_max_abs_smd       0.048836
analytic_se_of_smd         0.000749
observed_in_analytic_se   65.199669
null_max_mean              0.001462
null_max_largest           0.002115
permutation_p_value        0.000000
n_permutations            50.000000
dtype: float64

In [18]:
null_by_feature = permuted.groupby("feature")["abs_smd"].agg(
    null_mean="mean",
    null_p95=lambda values: values.quantile(0.95),
).reset_index()

comparison = balance[["feature", "abs_smd"]].merge(
    null_by_feature, on="feature")
comparison["observed_over_null_p95"] = (
    comparison["abs_smd"] / comparison["null_p95"])
comparison.sort_values("abs_smd", ascending=False)

,feature,abs_smd,null_mean,null_p95,observed_over_null_p95
0,f3,0.048836,0.000650,0.001519,32.148131
1,f6,0.040448,0.000717,0.001480,27.327868
2,f5,0.030557,0.000654,0.001404,21.763308
3,f9,0.024001,0.000635,0.001627,14.755616
4,f1,0.023986,0.000684,0.001492,16.077571
5,f8,0.022427,0.000653,0.001352,16.589430
6,f7,0.021269,0.000591,0.001513,14.060235
7,f10,0.010565,0.000549,0.001212,8.715643
8,f4,0.007972,0.000499,0.001227,6.498359
9,f0,0.006866,0.000656,0.001744,3.936937


If every feature exceeds its own null, the pattern is structural
rather than a fluke. Two explanations are consistent with that, and
anonymised features make it impossible to separate them:

- Assignment happened at a unit coarser than a single row, with
  features correlated to that unit.
- The features were measured during or after the campaign, so the ad
  itself nudged them.

The second would be more serious, since features affected by
treatment are contaminated inputs to an uplift model. Section 8 puts
a ceiling on how large either effect can be.

## 8. How large is it? Propensity and common support

Section 7 establishes that a relationship exists. It says nothing
about strength, and strength is what matters for modelling.

Train a classifier to predict `treatment` from the twelve features
and measure **AUC**: the probability it ranks a random treated user
above a random control user. 0.50 is chance, and it is unaffected by
the arms being unbalanced in size.

The predicted probabilities are **propensity scores**. Their spread
answers a second question. An uplift model compares treated users to
control users who look like them, so if any region of feature space
contains treated users and almost no control users, uplift there is
extrapolated rather than measured. The condition needed is called
**common support**.

At 5M rows the standard error of an AUC is around 0.0003, so 0.505
would be statistically distinguishable from 0.5 and mean nothing. The
same permutation approach supplies the comparison point.

In [19]:
sample = df.sample(n=min(PROPENSITY_SAMPLE_ROWS, len(df)),
                   random_state=PROPENSITY_SEED)
X = sample[FEATURES]
y_observed = sample[TREATMENT_COL].to_numpy()

print(f"rows: {len(sample):,}")
print(f"treated share: {y_observed.mean():.6f}")

rows: 5,000,000
treated share: 0.850007


In [20]:
def propensity_auc(features, labels, params, test_size, seed):
    """Fit a treatment classifier and score it on held-out rows.

    Args:
        features (pandas.DataFrame): Feature columns.
        labels (numpy.ndarray): Binary treatment labels.
        params (dict): LightGBM parameters.
        test_size (float): Held-out proportion.
        seed (int): Seed for the split.

    Returns:
        tuple: AUC, predicted propensities, and their labels.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=test_size,
        stratify=labels, random_state=seed)

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    propensity = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, propensity), propensity, y_test


auc_observed, propensity, y_test = propensity_auc(
    X, y_observed, LGB_PROPENSITY_PARAMS,
    PROPENSITY_TEST_SIZE, PROPENSITY_SEED)

print(f"observed AUC: {auc_observed:.5f}")

observed AUC: 0.50909


In [21]:
rng = np.random.default_rng(PROPENSITY_SEED)

null_aucs = []
for draw in range(N_NULL_DRAWS):
    permuted_labels = rng.permutation(y_observed)
    auc, _, _ = propensity_auc(
        X, permuted_labels, LGB_PROPENSITY_PARAMS,
        PROPENSITY_TEST_SIZE, PROPENSITY_SEED + draw + 1)
    null_aucs.append(auc)
    print(f"permuted labels, draw {draw}: AUC {auc:.5f}")

pd.Series({
    "auc_observed": auc_observed,
    "auc_null_mean": float(np.mean(null_aucs)),
    "auc_above_null": auc_observed - float(np.mean(null_aucs)),
})

permuted labels, draw 0: AUC 0.49975
permuted labels, draw 1: AUC 0.49962
permuted labels, draw 2: AUC 0.49985


auc_observed     0.509093
auc_null_mean    0.499738
auc_above_null   0.009355
dtype: float64

In [22]:
frame = pd.DataFrame({"propensity": propensity,
                      "treatment": y_test})

display(frame.groupby("treatment")["propensity"].describe(
    percentiles=[0.01, 0.50, 0.99]))

pd.Series({
    "base_rate": y_observed.mean(),
    "share_above_0.95": float((propensity > 0.95).mean()),
    "share_above_0.99": float((propensity > 0.99).mean()),
    "share_below_0.50": float((propensity < 0.50).mean()),
    "propensity_min": float(propensity.min()),
    "propensity_max": float(propensity.max()),
})

,count,mean,std,min,1%,50%,99%,max
treatment,,,,,,,,
0,"224,990.000000",0.849511,0.007424,0.721675,0.842038,0.848278,0.883791,0.970326
1,"1,275,010.000000",0.850140,0.009482,0.696183,0.842055,0.848404,0.896357,0.977837


base_rate          0.850007
share_above_0.95   0.000526
share_above_0.99   0.000000
share_below_0.50   0.000000
propensity_min     0.696183
propensity_max     0.977837
dtype: float64

Reading the propensity distribution against a base rate of 0.85:

| Score | Meaning |
|---|---|
| Near 0.85 | Both arms well represented around this user |
| Near 1.00 | Control users are scarce here. Uplift would be extrapolated |
| Near 0.00 | The mirror problem |

If the two arms have near-identical distributions, and the extremes
contain users from both, common support holds and uplift is
measurable everywhere.

The AUC also caps the contamination worry raised in section 7. If the
ad changed the features, that change is part of what makes the arms
distinguishable. Total distinguishability is the gap between the
observed AUC and the permuted null, so contamination cannot exceed
it.

## 9. What this notebook establishes

### Decisions carried into notebook 2

| Decision | Basis |
|---|---|
| Primary outcome is `visit` | Control-arm event counts in section 5 and the noise floor in section 6, two independent arguments |
| `conversion` reported, not modelled | At 50 bins the noise floor approaches the entire average effect |
| Intent to treat on `treatment` as assigned | Section 2b |
| `exposure` excluded from every feature set | Post-treatment and not randomised |
| Results reported at 20 bins | Section 6. Finer bins raise the noise floor for no gain |
| Any evaluation curve carries a random-ranking band | Section 6 |
| Analytic standard errors are trustworthy | `sd_ratio` near 1 throughout section 6 |
| Full 14M rows for reported results | No sampling is needed. LightGBM handles the file |
| Cost per targeted user reflects the 3.6% exposure rate | Section 2b. Notebook 3 depends on it |

### Limitations recorded here

1. The dataset is not a clean row-level randomised experiment.
   Section 7 quantifies the departure; section 8 bounds its
   consequence.
2. The features are anonymised, so whether they were measured before
   or after treatment cannot be determined.
3. The funnel decomposition in section 3 conditions on a
   post-treatment variable and is descriptive only.
4. Whether `exposure` is recorded before or after a visit within the
   campaign window is undocumented, which is a further reason the
   exposed-versus-unexposed comparison stays out of the analysis.

### Next

`02_uplift_models.ipynb` builds two uplift estimators, tests whether
calibration helps, and evaluates both against the noise floor
measured here.